https://www.kaggle.com/competitions/playground-series-s4e8

In [ ]:
from fastkaggle import *
from pathlib import Path
import zipfile
import pandas as pd
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [ ]:
comp_name = "playground-series-s4e8"

In [ ]:
setup_comp(comp_name)

In [ ]:
datapath = Path(comp_name)

In [ ]:
df_train = pl.read_csv(f'{datapath}/train.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.describe()

In [ ]:
df_train.dtypes

In [ ]:
df_train.select(pl.all().approx_n_unique())

In [ ]:
df_train.columns

Take a look at the value counts for the target 

In [ ]:
df_train['class'].value_counts()

Split the training set into data and target

In [ ]:
x = df_train.drop('class')
y = df_train['class']

For a less ordinal data we could use `One Hot Encoding`, but since the prediction is binary we would can use a basic mapping or sklearn's `LabelEncoder`

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
cat_cols = [col for col, dtype in x.schema.items() if dtype == pl.Utf8]
num_cols = [col for col in x.columns if col not in cat_cols]

In [ ]:
cat_cols

In [ ]:
num_cols

Create categorical transformer, handle NaN vals

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

Create numeric transformer, handle NaN vals

In [ ]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')) 
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

Split to training and test sets

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)

In [ ]:
y_train[:10]

Create a classifier

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=40, max_samples=200_000, random_state=42, verbose=2)),
    ]
)

Run fit 

In [ ]:
x_train_pandas = x_train.to_pandas()

In [ ]:
pipeline.fit(x_train_pandas, y_train)

In [ ]:
y_pred = pipeline.predict(x_test)

In [ ]:
y_pred.shape

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

In [ ]:
X_test = pl.read_csv(f'{datapath}/test.csv')

In [ ]:
preds = pipeline.predict(X_test)

In [ ]:
label_encoder.inverse_transform(preds)

In [ ]:
sample_df = pd.read_csv(datapath/'sample_submission.csv')

In [ ]:
sample_df.head()

In [ ]:
decoded_preds = label_encoder.inverse_transform(preds)

In [ ]:
sample_df['class'] = decoded_preds

In [ ]:
sample_df

In [ ]:
sample_df.to_csv('submission.csv', index=False)


In [ ]:
submit = True

In [ ]:
if not iskaggle and submit:
    from kaggle import api
    api.competition_submit_cli(file_name='submission.csv', message="baseline model", competition=comp_name)